# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string and OpenAI API key in Step 0, then run each cell in order. The notebook creates embeddings for the sample documents, stores them in Azure DocumentDB, creates vector and full-text indexes, and runs vector, BM25, fuzzy, phrase, and hybrid search.

> Full-text search in Azure DocumentDB is currently in gated preview. DiskANN vector search requires an M30 or higher cluster tier.


## Step 0: Connect and configure embeddings

This cell loads the MongoDB driver, accepts the DocumentDB connection string and OpenAI API key, and opens the workshop collection.

In [ ]:
let mongodb;
try { mongodb = require("mongodb"); } catch { require("child_process").execSync("npm install mongodb", { stdio: "inherit" }); mongodb = require("mongodb"); }
const { MongoClient } = mongodb;
const connectionString = process.env.DOCUMENTDB_CONNECTION_STRING || "<paste-your-azure-documentdb-connection-string-here>";
const openAiApiKey = process.env.OPENAI_API_KEY || "<paste-your-openai-api-key-here>";
const embeddingModel = process.env.OPENAI_EMBEDDING_MODEL || "text-embedding-3-small";
if (connectionString.includes("<paste")) throw new Error("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
if (openAiApiKey.includes("<paste")) throw new Error("Paste your OpenAI API key in this cell or set OPENAI_API_KEY.");
const client = new MongoClient(connectionString);
await client.connect();
const db = client.db("docdbworkshop");
const collection = db.collection("workshop_content");
await db.command({ ping: 1 });

## Step 1: Generate embeddings and load sample documents

This cell calls OpenAI to embed each sample document body, stores the vectors in `embedding`, and inserts the documents.

In [ ]:
async function createEmbedding(text) {
  const response = await fetch("https://api.openai.com/v1/embeddings", {
    method: "POST",
    headers: { "Authorization": `Bearer ${openAiApiKey}`, "Content-Type": "application/json" },
    body: JSON.stringify({ model: embeddingModel, input: text })
  });
  if (!response.ok) throw new Error(await response.text());
  const payload = await response.json();
  return payload.data[0].embedding;
}
const sourceDocs = [
  { _id: "doc-search-001", title: "DiskANN vector indexing", category: "vector", body: "Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.", sku: "SEARCH-VEC-001" },
  { _id: "doc-search-002", title: "BM25 keyword search", category: "full-text", body: "Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.", sku: "SEARCH-FTS-001" },
  { _id: "doc-search-003", title: "Hybrid search with RRF", category: "hybrid", body: "Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.", sku: "SEARCH-HYB-001" },
  { _id: "doc-search-004", title: "RAG grounding", category: "rag", body: "Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.", sku: "RAG-PIPE-001" }
];
await collection.drop().catch(() => {});
for (const doc of sourceDocs) doc.embedding = await createEmbedding(doc.body);
await collection.insertMany(sourceDocs);
const embeddingDimensions = sourceDocs[0].embedding.length;
({ loaded: await collection.countDocuments({}), embeddingDimensions });

## Step 2: Create vector and full-text indexes

The vector index dimension is taken from the generated OpenAI embedding.

In [ ]:
await db.command({ createIndexes: "workshop_content", indexes: [{ name: "idx_embedding_diskann", key: { embedding: "cosmosSearch" }, cosmosSearchOptions: { kind: "vector-diskann", dimensions: embeddingDimensions, similarity: "COS", maxDegree: 32, lBuild: 64 } }] });
await db.command({ createSearchIndexes: "workshop_content", indexes: [{ name: "idx_body_fts", definition: { mappings: { dynamic: false, fields: { body: { type: "string" } } } } }] });

## Step 3: Generate a query embedding and run vector search

The query text is embedded at runtime, then passed into `$search.cosmosSearch`.

In [ ]:
const searchText = "semantic retrieval for RAG";
const queryVector = await createEmbedding(searchText);
await collection.aggregate([{ $search: { cosmosSearch: { path: "embedding", vector: queryVector, k: 3 } } }, { $project: { _id: 0, title: 1, category: 1, score: { $meta: "searchScore" } } }]).toArray();

## Step 4: Run full-text, fuzzy, and phrase search

These queries use the BM25 search index and demonstrate exact keyword relevance, typo tolerance, and ordered phrase matching.

In [ ]:
const bm25 = await collection.aggregate([{ $search: { index: "idx_body_fts", text: { query: "BM25 ranking", path: "body" } } }, { $limit: 5 }, { $project: { _id: 0, title: 1, score: { $meta: "searchScore" } } }]).toArray();
const fuzzy = await collection.aggregate([{ $search: { index: "idx_body_fts", text: { query: "retrival augmentd genration", path: "body", fuzzy: { maxEdits: 1 } } } }, { $limit: 5 }, { $project: { _id: 0, title: 1, score: { $meta: "searchScore" } } }]).toArray();
const phrase = await collection.aggregate([{ $search: { index: "idx_body_fts", phrase: { query: "Reciprocal Rank Fusion", path: "body", slop: 0 } } }, { $limit: 5 }, { $project: { _id: 0, title: 1, score: { $meta: "searchScore" } } }]).toArray();
({ bm25, fuzzy, phrase });

## Step 5: Run hybrid search

Hybrid search embeds the query, retrieves BM25 and vector candidates, and fuses them with RRF.

In [ ]:
function rrf(lists, k = 60, topN = 5) {
  const scores = new Map(), docsById = new Map();
  for (const list of lists) list.forEach((doc, rank) => { const id = doc._id.toString(); docsById.set(id, doc); scores.set(id, (scores.get(id) ?? 0) + 1 / (k + rank + 1)); });
  return [...scores.entries()].sort((a, b) => b[1] - a[1]).slice(0, topN).map(([id, score]) => ({ ...docsById.get(id), rrfScore: score }));
}
const userQuery = "semantic retrieval for RAG";
const hybridVector = await createEmbedding(userQuery);
const keywordHits = await collection.aggregate([{ $search: { index: "idx_body_fts", text: { query: userQuery, path: "body" } } }, { $limit: 5 }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();
const vectorHits = await collection.aggregate([{ $search: { cosmosSearch: { path: "embedding", vector: hybridVector, k: 5 } } }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();
rrf([keywordHits, vectorHits]);